In [1]:
!pip install -q transformers datasets accelerate scikit-learn xgboost lightgbm catboost pandas numpy torch tqdm joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 74.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [2]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt
import seaborn as sns
import joblib

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Processing Stage 2 Benchmarking on Device: {str(DEVICE).upper()}")

Processing Stage 2 Benchmarking on Device: CUDA


In [3]:
DATA_PATH = "/kaggle/input/datasets/adhamashraf202200953/synthetic-code-dataset-for-plagiarism-detection/CoPlag-Contrastive-Dataset.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found at the specified path: {DATA_PATH}")

print("Loading dataset...")
df = pd.read_csv(DATA_PATH)

df['strat_key'] = df['language'] + "_" + df['label'].astype(str)
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['strat_key'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['strat_key'])

print(f"Data Splits Synced: Train ({len(train_df)}), Val ({len(val_df)}), Test ({len(test_df)})")

Loading dataset...
Data Splits Synced: Train (159395), Val (19924), Test (19925)


In [4]:
# =====================================================================
# 3. DEFINE LEXICAL FEATURE (JACCARD SIMILARITY)
# =====================================================================
def calculate_jaccard(code1, code2):
    set1 = set(str(code1).split())
    set2 = set(str(code2).split())
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0.0

# =====================================================================
# 4. FEATURE EXTRACTION PIPELINE FROM DEEP MODELS
# =====================================================================
PATH_CODEBERT = "/kaggle/input/notebooks/adhamashraf202200953/crossencoder-codebert/final_codebert_cross_model"
PATH_UNIXCODER = "/kaggle/input/notebooks/ashrafabdelhaq/crossencoder-unixcoder/final_unixcoder_cross_model"

def extract_semantic_features(dataframe, desc="Processing"):
    print(f"\nExtracting Deep Semantic Features [{desc}]...")
    
    tok_bert = AutoTokenizer.from_pretrained(PATH_CODEBERT)
    mod_bert = AutoModelForSequenceClassification.from_pretrained(PATH_CODEBERT).to(DEVICE).eval()
    
    tok_unix = AutoTokenizer.from_pretrained(PATH_UNIXCODER)
    mod_unix = AutoModelForSequenceClassification.from_pretrained(PATH_UNIXCODER).to(DEVICE).eval()
    
    features_list = []
    
    with torch.no_grad():
        for idx, row in tqdm(dataframe.iterrows(), total=len(dataframe), desc=desc):
            c1, c2 = str(row['code_a']), str(row['code_b'])
            
            inputs_bert = tok_bert(c1, c2, max_length=512, padding="max_length", truncation=True, return_tensors="pt").to(DEVICE)
            outputs_bert = mod_bert(**inputs_bert)
            proba_bert = F.softmax(outputs_bert.logits, dim=-1)[0][1].item() 
            
            inputs_unix = tok_unix(c1, c2, max_length=512, padding="max_length", truncation=True, return_tensors="pt").to(DEVICE)
            outputs_unix = mod_unix(**inputs_unix)
            proba_unix = F.softmax(outputs_unix.logits, dim=-1)[0][1].item() 
            
            jaccard = calculate_jaccard(c1, c2)
            
            features_list.append({
                "proba_codebert": proba_bert,
                "proba_unixcoder": proba_unix,
                "lexical_jaccard": jaccard,
                "target_label": int(row['label'])
            })
            
    del mod_bert, mod_unix
    torch.cuda.empty_cache()
    gc.collect()
    
    return pd.DataFrame(features_list)

In [5]:
meta_train = extract_semantic_features(val_df, desc="Building Meta-Train Set")
meta_test = extract_semantic_features(test_df, desc="Building Meta-Test Set")

X_train = meta_train.drop(columns=['target_label'])
y_train = meta_train['target_label']

X_test = meta_test.drop(columns=['target_label'])
y_test = meta_test['target_label']

# =====================================================================
# 5. DEFINE & TRAIN THE 4 COMPETITOR MODELS
# =====================================================================
models = {
    "XGBoost": XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=150, depth=4, learning_rate=0.05, random_seed=42, verbose=0),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42, n_jobs=-1)
}

benchmarking_results = []
trained_models = {}

print("\nStarting Meta-Classifiers Competitive Training Tournament...")


Extracting Deep Semantic Features [Building Meta-Train Set]...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Building Meta-Train Set: 100%|██████████| 19924/19924 [23:20<00:00, 14.22it/s]



Extracting Deep Semantic Features [Building Meta-Test Set]...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Building Meta-Test Set: 100%|██████████| 19925/19925 [23:26<00:00, 14.17it/s]



Starting Meta-Classifiers Competitive Training Tournament...


In [6]:
for name, model in models.items():
    print(f" -> Training Model: [{name}] ...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    preds = model.predict(X_test)
    
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds)
    
    benchmarking_results.append({
        "Model Name": name,
        "Test Accuracy": acc,
        "Test F1-Score": f1,
        "Test Precision": prec,
        "Test Recall": rec
    })

results_df = pd.DataFrame(benchmarking_results).sort_values(by="Test F1-Score", ascending=False).reset_index(drop=True)
print("\nBENCHMARKING TOURNAMENT LEADERBOARD SHEET:")
print(results_df.to_string(index=False))

 -> Training Model: [XGBoost] ...
 -> Training Model: [LightGBM] ...
 -> Training Model: [CatBoost] ...
 -> Training Model: [Random Forest] ...

BENCHMARKING TOURNAMENT LEADERBOARD SHEET:
   Model Name  Test Accuracy  Test F1-Score  Test Precision  Test Recall
Random Forest       0.810238       0.839509        0.727293     0.992672
      XGBoost       0.810088       0.839211        0.727601     0.991267
     CatBoost       0.809787       0.838874        0.727581     0.990363
     LightGBM       0.809586       0.838786        0.727233     0.990765


In [7]:
# =====================================================================
# 6. VISUALIZE COMPARATIVE CHARTS & SAVE EXPERIMENT METRICS
# =====================================================================
print("\nConstructing Comparison and Performance Visualizations...")

plt.figure(figsize=(12, 6))
melted_results = pd.melt(results_df, id_vars="Model Name", value_vars=["Test Accuracy", "Test F1-Score"], var_name="Metric", value_name="Score")
sns.barplot(data=melted_results, x="Model Name", y="Score", hue="Metric", palette="viridis")
plt.ylim(0.70, 1.0) 
plt.title("Stage 2: Meta-Classifiers Performance Comparison")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig("/kaggle/working/meta_models_comparison.png", dpi=300, bbox_inches='tight')
plt.close()

champion_name = results_df.iloc[0]["Model Name"]
champion_model = trained_models[champion_name]
champion_preds = champion_model.predict(X_test)

print(f"\nSaving the Champion Model [{champion_name}] into Production Artifacts...")
joblib.dump(champion_model, f"/kaggle/working/final_champion_{champion_name.lower().replace(' ', '_')}_model.pkl")

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, champion_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=["Predicted Clean", "Predicted Plagiarized"],
            yticklabels=["Actual Clean", "Actual Plagiarized"])
plt.title(f"Final Champion Layer ({champion_name}) Confusion Matrix")
plt.ylabel("True Class")
plt.xlabel("Predicted Class")
plt.savefig("/kaggle/working/champion_confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.close()

if hasattr(champion_model, "feature_importances_"):
    plt.figure(figsize=(10, 5))
    feat_importances = pd.Series(champion_model.feature_importances_, index=X_train.columns)
    feat_importances.nlargest(10).plot(kind='barh', color='indigo')
    plt.title(f"{champion_name} Feature Importance Profile")
    plt.xlabel("Relative Weight Score")
    plt.savefig("/kaggle/working/champion_feature_importance.png", dpi=300, bbox_inches='tight')
    plt.close()

print("\nSuccess! All 4 models evaluated. Leaderboard rendered, plots drawn, and the champion model is securely saved as a '.pkl' file.")


Constructing Comparison and Performance Visualizations...

Saving the Champion Model [Random Forest] into Production Artifacts...

Success! All 4 models evaluated. Leaderboard rendered, plots drawn, and the champion model is securely saved as a '.pkl' file.
